In [8]:
from datetime import datetime
import numpy as np
import xarray as xr
import os
os.environ["SATPY_CONFIG_PATH"] = "/wolke_scratch/dnikolo/MSGI/satpy_plugin"

from satpy import Scene, available_readers
print("claas3_cpp" in available_readers())  # should print True
from pyresample.geometry import AreaDefinition
from pyresample import utils as pr_utils

True


In [ ]:
available_readers()

In [9]:

# ---- helper: most-frequent (mode) for integer flags ----
def _mode_reduce(values):
    """values: 1D array of numbers (masked already); returns int or fill."""
    if values.size == 0:
        return np.nan
    # bincount wants non-negative ints; treat -1/fill as missing
    vals = values[np.isfinite(values)].astype(np.int64)
    if vals.size == 0:
        return np.nan
    # If you want ties to prefer "liquid" (1) over others, you can bias here.
    counts = np.bincount(vals)
    return np.argmax(counts)

In [10]:


# ---- point Satpy to your custom reader ----
# Make sure the folder holding claas3_cpp.yaml is in SATPY_CONFIG_PATH,
# e.g.: export SATPY_CONFIG_PATH=/path/to/your/yaml

# List of input files (you can pass a single path or many)
files = [
    "/wolke_scratch/dnikolo/CLAAS_Data/np/2009/02/03/CPPin20090203004500405SVMSG01MD.nc",
    # add more files if you want to aggregate time periods before resampling
    # "/wolke_scratch/dnikolo/CLAAS_Data/np/2009/02/03/CPPin20090203000000405SVMSG01MD.nc",
    # "/wolke_scratch/dnikolo/CLAAS_Data/np/2009/02/05/CPPin20090205000000405SVMSG01MD.nc",
    # "/wolke_scratch/dnikolo/CLAAS_Data/np/2009/02/05/CPPin20090205180000405SVMSG01MD.nc",
]

# Create scene and load only cph
scn = Scene(filenames=files, reader="claas3_cpp")
scn.load(["cph"])  # add "cph_ext" etc. if desired

# Ensure cph is integer-like for a proper categorical mode
# (your file stores int16 with scale_factor=1, so this typically does nothing)
cph = scn["cph"]
# Replace fill (-1) with NaN for the reducer
fill_value = -1
cph = cph.where(cph != fill_value)

# Build a 1-degree global target AreaDefinition (EPSG:4326 regular grid)
# Here we make a grid from -90..+90, -180..+180 at 1° resolution.
# pyresample AreaDefinition needs proj dict / proj4 string; use EPSG:4326 lon-lat with pixel size 1°.
# We’ll define it by extents & shape:
lat_res = 1.0
lon_res = 1.0
nlat = int(180/lat_res) + 1  # 181 rows (-90..+90 inclusive)
nlon = int(360/lon_res) + 1  # 361 cols (-180..+180 inclusive)

# Area extent for lon/lat in degrees: (min_lon, min_lat, max_lon, max_lat)
area_extent = (-180.0, -90.0, 180.0, 90.0)
area_id = "global_1deg"
area_name = "Global LatLon 1deg"
proj_id = "latlon"
proj_dict = {"proj": "longlat", "datum": "WGS84"}

target_area = AreaDefinition(
    area_id, area_name, proj_id, proj_dict, nlon, nlat, area_extent
)

# Resample with 'bucket' (categorical mode per 1° cell).
# Satpy's bucket passes data to pyresample.bucket. We provide our reducer.
# Some Satpy versions call it "aggregation_function"; others "reduce_func".
# We support both; only one will be used depending on version.
kwargs = {
    "resampler": "bucket",
    "reduce_data": True,
    "fill_value": np.nan,
}
# try both kw names for the reducer:
try:
    rs = scn.resample(
        target_area, **kwargs, aggregation_function=_mode_reduce
    )
except TypeError:
    rs = scn.resample(
        target_area, **kwargs, reduce_func=_mode_reduce
    )

# The result is a new Scene with 1° grid; get the resampled cph
cph_1deg = rs["cph"]

# Write to NetCDF/GeoTIFF/PNG as you wish; here NetCDF:
cph_1deg.attrs["long_name"] = "Cloud Thermodynamic Phase (mode in 1x1 deg)"
rs.save_datasets(writer="cf", datasets=["cph"], filename="cph_1deg_bucket.nc")
print("Saved: cph_1deg_bucket.nc")


INFO:satpy.readers.core.loading:Cannot use ['/wolke_scratch/dnikolo/MSGI/satpy_plugin/readers/claas3_cpp.yaml']


ValueError: No supported files found